<a href="https://colab.research.google.com/github/Jadhav-Suraj/wordCom-Ai/blob/main/LSTM_and_GRU(Ai_school).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('qoute_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
quotes = df['quote']

In [ ]:
quotes.head()

In [ ]:
## preprocessing the data
## 1. lowercase all
quotes = quotes.str.lower()

In [ ]:
## 2. remove punctuation
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [ ]:
quotes.head()

In [ ]:
## Applying Tokenization(words or sentences to number)
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
## setting vocab_size

vocab_size = 8978
tok = Tokenizer(num_words=vocab_size)
tok.fit_on_texts(quotes)

In [ ]:
word_index = tok.word_index
print(len(word_index))
list(word_index.items())[:10]

In [ ]:
sequence = tok.texts_to_sequences(quotes)

In [ ]:
quotes[0]

In [ ]:
sequence[0]

In [ ]:
## splitting the data in parts so we use first single words then two and like increasing the words
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [ ]:
len(X)

In [ ]:
## maximum sentence length and apply adding for same scale of inputs
maxlen = max([len(x) for x in X])
maxlen

### where numbers are not present and we want to same input features so we apply padding where we add zeros to where the null values

In [ ]:
## for same size inputs we apply padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen=maxlen,padding='pre')

In [ ]:
X_padded[0]

In [ ]:
X[0]

In [ ]:
## converting into array(output)
y = np.array(y)

In [ ]:
X_padded.shape

In [ ]:
## applying one hot encoding to output variable
from tensorflow.keras.utils import to_categorical

y_one_hot = to_categorical(y, num_classes=vocab_size)

In [ ]:
y_one_hot.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [ ]:
## fistly creating embeddings

embedding_dim = 80
rnn_units = 128 ## hidden layers neurons

In [ ]:
## RNN Model

rnn_model = Sequential()
rnn_model.add(
    Embedding(
        input_dim=vocab_size,output_dim=embedding_dim,input_length=maxlen
        )
    )
rnn_model.add(SimpleRNN(units=rnn_units,activation='relu'))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))

In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'])

In [ ]:
rnn_model.summary()

In [ ]:
## Creating LSTM Model

lstm_model = Sequential()
lstm_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=maxlen
        )
    )
lstm_model.add(LSTM(units=rnn_units,activation='relu'))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lstm_model.summary()

In [ ]:
# ## RNN model and their history

# epochs = 10
# batch_size = 128

# history_rnn = rnn_model.fit(
#     X_padded,
#     y_one_hot,
#     epochs=epochs,
#     batch_size=batch_size,
#     validation_split=0.1
# )

In [ ]:
## LSTM Model and their history

epochs = 100
batch_size = 128

history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

In [ ]:
## saving model in h5 format used for deep learning models (it is like pickle format file for the machine leraning we use)
## but in modern era we use '.keras' for better understanding

lstm_model.save('lstm_model.keras')

In [ ]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def predictor(model,tok,text,maxlen):
  text = text.lower()

  seq = tok.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen=maxlen,padding='pre')

  pred = model.predict(seq)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [ ]:
seed_text = "you will"
next_word = predictor(lstm_model,tok,seed_text,maxlen)
print(next_word)

In [ ]:
def generate_text(model, tok, seed_text, maxlen, num_words):
  for _ in range(num_words):
    next_word = predictor(model, tok, seed_text, maxlen)
    if next_word is None:
      break
    seed_text += " " + next_word
  return seed_text


In [ ]:
## checking for more number of words prediction

seed = "the meaning of life"
generate_text = generate_text(lstm_model, tok, seed, maxlen, 10)
print(generate_text)

In [ ]:
## Storing tokenizer in pickle file
import pickle

with open("tokenizer.pkl","wb") as f:
  pickle.dump(tok,f)


In [ ]:
## storing maxlen in pickle file

with open("max_len.pkl","wb") as f:
  pickle.dump(maxlen,f)